# 05 · Camada Silver — Limpeza, Regras de Negócio e Features de ML

Este notebook lê a Bronze (`bronze.incidentes`) e grava **quatro tabelas
Delta relacionadas** no schema `silver`, decisão tomada deliberadamente
para permitir features personalizadas por modelo, sem forçar um único
grão para tudo:

| Tabela | Grão | Uso |
|---|---|---|
| `silver.incidentes_tratados` | 1 linha por incidente | Base limpa e tipada, **sem features de ML** — alimenta a Gold |
| `silver.calendario_feriados` | 1 linha por feriado | Tabela de referência (não é feature) — nome do feriado nacional, usada pela Gold em `dim_data` |
| `silver.features_calendario` | 1 linha por dia | Atributos de calendário (dia da semana, trimestre, fim de semana, feriado...), reutilizável por qualquer segmentação |
| `silver.features_series_produto` | 1 linha por (dia, produto, prioridade) | Lags/médias móveis/target para modelos segmentados por Produto × Prioridade |
| `silver.features_series_categoria` | 1 linha por (dia, categoria, prioridade) | Mesma lógica, segmentada por Categoria × Prioridade |
| `silver.features_series_prioridade` | 1 linha por (dia, prioridade) | Tendência pura por prioridade (P2/P3) — o recorte que a Locaweb pede explicitamente, sem misturar com produto/categoria |

As tabelas de série temporal se relacionam com `features_calendario` pela
chave `data_abertura`, e com `incidentes_tratados` pelas colunas de
segmentação (`produto`, `categoria`) — não duplicamos os atributos de
calendário dentro de cada tabela de série, evitando redundância.

Todas as decisões de tratamento aqui (o que virou flag, o que ficou nulo
de propósito, por que o filtro de KPI) vêm diretamente dos achados do
notebook `04_exploratory_analysis` — não repetimos a análise aqui, só
aplicamos as decisões.

## Dependência: `holidays` (calendário nacional de feriados BR)

Usado só para enriquecer `features_calendario` com `is_feriado`/
`nome_feriado` — ver a seção da Tabela 2 mais abaixo para o detalhe de
cobertura (feriados nacionais automáticos; municipais/regionais ficam
de fora, precisam de importação manual futura).

In [0]:
%pip install -q holidays
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./00_config

# 00 · Configuração do projeto AntecipeAI

Este notebook **não é uma etapa do pipeline** — ele é chamado com `%run` no
início de todos os outros notebooks para carregar a configuração central
do projeto a partir do arquivo `.env`.

A ideia por trás disso: migrar o projeto do Databricks Free (tudo managed,
storage do próprio metastore) para um ambiente de nuvem (S3/AWS,
ADLS/Azure, GCS/GCP, Object Storage/OCI) deve ser uma **troca de valores
no `.env`**, e não uma reescrita de notebook.

## Localizar e carregar o `.env`

Assumimos a estrutura de pastas `antecipeai/notebooks/` e
`antecipeai/config/antecipeai.env` lado a lado no Repo/Workspace. Se a sua
estrutura for diferente, informe o caminho exato no widget
`env_file_path` antes de rodar este notebook.

Configuração carregada de: ../config/antecipeai.env


## Defaults (usados apenas se o `.env` não for encontrado)

## Variáveis expostas para os notebooks que derem `%run` neste

Por ser chamado via `%run`, tudo que é definido aqui fica disponível no
notebook que chamou — não precisa importar nada manualmente depois.

Configuração ativa:
  CATALOG         = antecipeai
  SCHEMA_LANDING  = landing
  SCHEMA_BRONZE   = bronze
  SCHEMA_SILVER   = silver
  SCHEMA_GOLD     = gold
  VOLUME_RAW      = raw
  TABLE_TYPE      = MANAGED
  STORAGE_ROOT    = (vazio - ok p/ MANAGED)
  CLOUD_PROVIDER  = NONE


## Helper: DDL de criação de schema (MANAGED vs EXTERNAL)

Centraliza a única parte do projeto que de fato muda entre "Databricks
Free" e "produção na nuvem": **onde** o schema grava fisicamente os
dados. O resto do código (leituras, transformações, escrita de tabelas)
não muda uma linha.

Helpers disponíveis: qualified_table(schema, table), volume_path(subpath), create_schema_sql(schema_name)


In [0]:
from pyspark.sql import functions as F, Window

bronze_table = qualified_table(SCHEMA_BRONZE, "incidentes")
bronze = spark.table(bronze_table)

print(f"Lendo {bronze_table}: {bronze.count()} linhas")

Lendo antecipeai.bronze.incidentes: 122543 linhas


## Tabela 1 · `silver.incidentes_tratados`

Tipagem completa + as flags de qualidade decididas na EDA:
- `entrou_kpi_flag_fonte` / `entrou_kpi_flag_calculada` / `kpi_regra_divergente`
— mantemos as duas versões da flag de KPI, sem sobrescrever a origem
(achado: 151 casos divergentes, concentrados em dez/2025).
- `duracao_suspeita` — incidentes com duração > 10x o SLA da prioridade
mas não marcados como violação (achado: 2.499 casos). Fica como flag,
não é excluído nem "corrigido".
- `produto`/`categoria` nulos viram `NAO_CLASSIFICADO` explícito (não
apagamos a informação de que o incidente não tinha classificação —
isso é estrutural de tickets abertos por monitoramento).

In [0]:
sla_expr = (
    F.when(F.col("prioridade_num").isin(1, 2), 4 * 3600)
    .when(F.col("prioridade_num") == 3, 12 * 3600)
    .when(F.col("prioridade_num") == 4, 24 * 3600)
    .when(F.col("prioridade_num") == 5, 96 * 3600)
)

_staged = (
    bronze
    .withColumn("prioridade_num", F.regexp_extract("Prioridade", r"^(\d)", 1).cast("int"))
    .withColumn("prioridade_desc", F.trim(F.regexp_extract("Prioridade", r"-\s*(.*)$", 1)))
    .withColumn("dt_aberto", F.to_timestamp("Aberto"))
    .withColumn("dt_resolvido", F.to_timestamp("Resolvido"))
    .withColumn("dt_encerrado", F.to_timestamp("Encerrado"))
    .withColumn("data_abertura", F.to_date("dt_aberto"))
    .withColumn("duracao_segundos", F.col("Duração").cast("long"))
    .withColumn("tem_incidente_pai", F.col("Incidente Pai").isNotNull())
    .withColumn("entrou_kpi_flag_fonte", F.col("Entrou para KPI?") == "SIM")
    .withColumn("kpi_violado_fonte", F.col("KPI Violado?") == "SIM")
    .withColumn("produto", F.coalesce(F.col("Produto"), F.lit("NAO_CLASSIFICADO")))
    .withColumn("categoria", F.coalesce(F.col("Categoria"), F.lit("NAO_CLASSIFICADO")))
)

_staged = (
    _staged
    .withColumn(
        "entrou_kpi_flag_calculada",
        _staged["prioridade_num"].isin(1, 2, 3)
        & ~_staged["tem_incidente_pai"]
        & (_staged["Status"] != "Sem Intervenção"),
    )
    .withColumn("kpi_regra_divergente", F.col("entrou_kpi_flag_fonte") != F.col("entrou_kpi_flag_calculada"))
    .withColumn("sla_segundos", sla_expr)
    .withColumn(
        "duracao_suspeita",
        F.col("entrou_kpi_flag_fonte") & (F.col("duracao_segundos") > F.col("sla_segundos") * 10),
    )
)

## Período coberto pelo histórico — calculado, não hardcoded

`DATA_MIN`/`DATA_MAX` vêm do próprio dado (`MIN`/`MAX` de
`data_abertura`), não de uma data cravada no código. Isso evita perda
silenciosa: se uma extração futura da Locaweb trouxer incidentes fora
de um range hardcoded, essas linhas simplesmente sumiriam das tabelas de
série temporal sem erro nenhum. Aqui, se o dado mudar, os notebooks
acompanham sozinhos.

Essas datas cobrem só o **histórico real** — usadas para treinar (Silver).
A `Gold` (`dim_data`, notebook `06`) estende um pouco mais para o
futuro, para caber os horizontes de previsão D+1/D+7 — não fazemos essa
extensão aqui, porque isso injetaria dias fictícios com
`qtd_incidentes = 0` nas features de treino, o que seria dado falso
(zero incidentes é diferente de "ainda não aconteceu").

In [0]:
_periodo = _staged.agg(F.min("data_abertura").alias("min"), F.max("data_abertura").alias("max")).first()
DATA_MIN = _periodo["min"].isoformat()
DATA_MAX = _periodo["max"].isoformat()

print(f"Período coberto pelo histórico: {DATA_MIN} a {DATA_MAX}")

incidentes_tratados = _staged.select(
    F.col("Número").alias("numero_incidente"),
    "prioridade_num",
    "prioridade_desc",
    "produto",
    "categoria",
    F.col("Subcategoria").alias("subcategoria"),
    F.col("Grupo designado").alias("grupo_designado"),
    F.col("Item de configuração").alias("item_configuracao"),
    "dt_aberto",
    "dt_resolvido",
    "dt_encerrado",
    "data_abertura",
    "duracao_segundos",
    "sla_segundos",
    F.col("Código de fechamento").alias("codigo_fechamento"),
    F.col("Descrição resumida").alias("descricao_resumida"),
    F.col("Solução").alias("solucao"),
    F.col("Aberto por").alias("aberto_por"),
    "tem_incidente_pai",
    F.col("Status").alias("status"),
    "entrou_kpi_flag_fonte",
    "entrou_kpi_flag_calculada",
    "kpi_regra_divergente",
    "kpi_violado_fonte",
    "duracao_suspeita",
)

(
    incidentes_tratados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_SILVER, "incidentes_tratados"))
)

print(f"silver.incidentes_tratados gravada: {incidentes_tratados.count()} linhas")

Período coberto pelo histórico: 2023-01-02 a 2025-12-31
silver.incidentes_tratados gravada: 122543 linhas


## Tabela 2 · `silver.features_calendario`

Grão diário, cobrindo o período completo do histórico
(`2023-01-02` a `2025-12-31`). Serve de base para o cross-join das
tabelas de série temporal (garante que todo dia exista, mesmo sem
incidentes — essencial para não quebrar cálculo de lag/médias móveis)
e pode ser usada por qualquer modelo que precise de atributos de
calendário sem duplicar essa lógica em cada tabela de série.

**`is_feriado`** (0/1) vem da lib `holidays`, calendário nacional
brasileiro — cobertura automática, sem depender de importação manual.
**Limitação:** só feriados nacionais/federais. Feriados municipais ou
estaduais (ex.: aniversário de cidade) não são cobertos por essa lib —
se algum desses importar para o modelo, precisam ser importados à parte
e mesclados aqui depois.

**Nota de design:** o nome do feriado (`nome_feriado`) é dado
categórico/descritivo, não uma feature pronta para modelo — por isso
**não entra nesta tabela**. Ele vive separado, em
`silver.calendario_feriados` (tabela de referência, não de features),
de onde a Gold busca para popular `gold.dim_data`. Mantém
`features_calendario` estritamente numérica/ML-ready, como as demais
tabelas `features_*` deste notebook.

In [0]:
import holidays

anos_cobertos = list(range(int(DATA_MIN[:4]), int(DATA_MAX[:4]) + 1))
br_feriados = holidays.Brazil(years=anos_cobertos)

feriados_rows = [(d.isoformat(), nome) for d, nome in sorted(br_feriados.items())]
feriados_df = (
    spark.createDataFrame(feriados_rows, ["data_abertura", "nome_feriado"])
    .withColumn("data_abertura", F.col("data_abertura").cast("date"))
)

print(f"Feriados nacionais BR gerados ({anos_cobertos[0]}-{anos_cobertos[-1]}): {feriados_df.count()}")

# Tabela de referência (NÃO é uma tabela de features) — só o nome do
# feriado, só nos dias em que ele ocorre. É daqui, não de
# features_calendario, que a Gold busca nome_feriado para dim_data.
(
    feriados_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_SILVER, "calendario_feriados"))
)
print(f"silver.calendario_feriados gravada: {feriados_df.count()} linhas")

Feriados nacionais BR gerados (2023-2025): 29
silver.calendario_feriados gravada: 29 linhas


In [0]:
calendario_base = (
    spark.range(1)
    .select(F.explode(F.sequence(F.lit(DATA_MIN).cast("date"), F.lit(DATA_MAX).cast("date"))).alias("data_abertura"))
)

features_calendario = (
    calendario_base
    .withColumn("ano", F.year("data_abertura"))
    .withColumn("mes", F.month("data_abertura"))
    .withColumn("dia", F.dayofmonth("data_abertura"))
    .withColumn("trimestre", F.quarter("data_abertura"))
    .withColumn("dia_semana_num", F.dayofweek("data_abertura"))
    .withColumn("dia_semana_nome", F.date_format("data_abertura", "EEEE"))
    .withColumn("is_fim_de_semana", F.col("dia_semana_num").isin(1, 7))
    .withColumn("semana_do_ano", F.weekofyear("data_abertura"))
    .join(F.broadcast(feriados_df.select("data_abertura")).withColumn("is_feriado_tmp", F.lit(1)), "data_abertura", "left")
    .withColumn("is_feriado", F.coalesce(F.col("is_feriado_tmp"), F.lit(0)))
    .drop("is_feriado_tmp")
)

(
    features_calendario.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_SILVER, "features_calendario"))
)

qtd_feriados_no_periodo = features_calendario.filter(F.col("is_feriado") == 1).count()
print(f"silver.features_calendario gravada: {features_calendario.count()} linhas (esperado: 1.095 dias)")
print(f"Dias marcados como feriado no período: {qtd_feriados_no_periodo}")

silver.features_calendario gravada: 1095 linhas (esperado: 1.095 dias)
Dias marcados como feriado no período: 28


## Tabelas 3, 4 e 5 · Séries temporais segmentadas (Produto, Categoria, Prioridade)

**Decisão de arquitetura (achado da EDA item 10):** a variável-alvo é a
contagem diária de incidentes que **entram no KPI**
(`entrou_kpi_flag_fonte = true`) — não o volume bruto, que tem uma
mudança de regime em set/2025 causada por uma ferramenta de
monitoramento, não por variação operacional real.

**`prioridade_num` entrou como segmentação em `produto`/`categoria`, e
ganhou tabela própria** (`features_series_prioridade`) — a Locaweb pede
tendência especificamente por prioridade (P2/P3 obrigatórias), e isso
não é a mesma coisa que olhar produto/categoria: um produto pode ter
volume estável enquanto sua fatia de P2 cresce, por exemplo.

A função abaixo é parametrizada por uma **lista** de colunas de
segmentação (não mais uma só), para não duplicar a lógica entre os três
recortes — e para deixar fácil adicionar um quarto no futuro (ex.:
`grupo_designado`) sem reescrever nada.

In [0]:
def build_time_series_features(colunas_segmentacao: list) -> "DataFrame":
    """
    Constrói uma tabela de série temporal diária (grão: data + colunas de
    segmentação) com lags, médias móveis e os targets D+1/D+7, a partir de
    incidentes elegíveis para KPI. Preenche combinações sem incidentes com
    zero (essencial para lag/rolling não quebrarem).

    Aceita 1+ colunas de segmentação — ex.: ["produto", "prioridade_num"]
    para tendência por produto quebrada por prioridade, ou apenas
    ["prioridade_num"] para a tendência pura de P2/P3 que a Locaweb pede
    explicitamente no desafio.
    """
    base_kpi = _staged.filter(F.col("entrou_kpi_flag_fonte"))

    agg = (
        base_kpi.groupBy("data_abertura", *colunas_segmentacao)
        .agg(F.count("*").alias("qtd_incidentes"))
        .repartition(16)
    )

    valores_segmento = agg.select(*colunas_segmentacao).distinct()
    grid = calendario_base.crossJoin(valores_segmento).repartition(16)
    full = grid.join(agg, ["data_abertura", *colunas_segmentacao], "left").fillna(0, subset=["qtd_incidentes"])

    w = Window.partitionBy(*colunas_segmentacao).orderBy("data_abertura")
    w7 = Window.partitionBy(*colunas_segmentacao).orderBy("data_abertura").rowsBetween(-6, 0)
    w14 = Window.partitionBy(*colunas_segmentacao).orderBy("data_abertura").rowsBetween(-13, 0)

    feat = (
        full
        .withColumn("lag_1d", F.lag("qtd_incidentes", 1).over(w))
        .withColumn("lag_7d", F.lag("qtd_incidentes", 7).over(w))
        .withColumn("lag_14d", F.lag("qtd_incidentes", 14).over(w))
        .withColumn("media_movel_7d", F.round(F.avg("qtd_incidentes").over(w7), 2))
        .withColumn("media_movel_14d", F.round(F.avg("qtd_incidentes").over(w14), 2))
        # targets: o que queremos prever para D+1 e D+7 a partir da linha atual
        .withColumn("target_d1", F.lead("qtd_incidentes", 1).over(w))
        .withColumn("target_d7", F.lead("qtd_incidentes", 7).over(w))
    )
    return feat


# Grão: dia + produto + prioridade — permite treinar "previsão de P2 do
# produto X", não só volume geral do produto.
features_series_produto = build_time_series_features(["produto", "prioridade_num"])
(
    features_series_produto.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_SILVER, "features_series_produto"))
)
print(f"silver.features_series_produto gravada: {features_series_produto.count()} linhas")

# Grão: dia + categoria + prioridade — mesma lógica, outro recorte.
features_series_categoria = build_time_series_features(["categoria", "prioridade_num"])
(
    features_series_categoria.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_SILVER, "features_series_categoria"))
)
print(f"silver.features_series_categoria gravada: {features_series_categoria.count()} linhas")

# Grão: dia + prioridade, SEM outro corte — a tendência "pura" de P2/P3 que
# o desafio da Locaweb pede explicitamente (não é a mesma coisa que somar as
# duas tabelas acima por produto/categoria).
features_series_prioridade = build_time_series_features(["prioridade_num"])
(
    features_series_prioridade.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_table(SCHEMA_SILVER, "features_series_prioridade"))
)
print(f"silver.features_series_prioridade gravada: {features_series_prioridade.count()} linhas")

silver.features_series_produto gravada: 87600 linhas
silver.features_series_categoria gravada: 216810 linhas
silver.features_series_prioridade gravada: 2190 linhas


## Conferência final

In [0]:
for t in ["incidentes_tratados", "calendario_feriados", "features_calendario", "features_series_produto", "features_series_categoria", "features_series_prioridade"]:
    full_name = qualified_table(SCHEMA_SILVER, t)
    print(f"{full_name}: {spark.table(full_name).count()} linhas")

display(spark.table(qualified_table(SCHEMA_SILVER, "incidentes_tratados")).limit(5))

antecipeai.silver.incidentes_tratados: 122543 linhas
antecipeai.silver.calendario_feriados: 29 linhas
antecipeai.silver.features_calendario: 1095 linhas
antecipeai.silver.features_series_produto: 87600 linhas
antecipeai.silver.features_series_categoria: 216810 linhas
antecipeai.silver.features_series_prioridade: 2190 linhas


numero_incidente,prioridade_num,prioridade_desc,produto,categoria,subcategoria,grupo_designado,item_configuracao,dt_aberto,dt_resolvido,dt_encerrado,data_abertura,duracao_segundos,sla_segundos,codigo_fechamento,descricao_resumida,solucao,aberto_por,tem_incidente_pai,status,entrou_kpi_flag_fonte,entrou_kpi_flag_calculada,kpi_regra_divergente,kpi_violado_fonte,duracao_suspeita
INC8654273,3,Média,NAO_CLASSIFICADO,NAO_CLASSIFICADO,null,Team14,IC00001,2025-12-31T23:45:18.000Z,null,2025-12-31T23:45:32.000Z,2025-12-31,14,43200,null,Problem: Apache Busy Workers,null,Monitoramento,false,Sem Intervenção,false,false,false,null,false
INC8654270,4,Baixa,NAO_CLASSIFICADO,NAO_CLASSIFICADO,null,Team14,IC00002,2025-12-31T23:39:36.000Z,null,2025-12-31T23:43:05.000Z,2025-12-31,209,86400,null,Problem: Check Application Monitoring,null,Monitoramento,false,Sem Intervenção,false,false,false,null,false
INC8654264,4,Baixa,NAO_CLASSIFICADO,NAO_CLASSIFICADO,null,Team14,null,2025-12-31T23:23:10.000Z,null,2025-12-31T23:25:00.000Z,2025-12-31,110,86400,null,"Problem: Alarm Application Monitoring database Message: HTTPSConnectionPool(host='centraldoclienteIC04172locawebIC04172comIC04172br', port=443): Read timed out. (read timeout=25)",null,Monitoramento,false,Sem Intervenção,false,false,false,null,false
INC8654263,4,Baixa,NAO_CLASSIFICADO,NAO_CLASSIFICADO,null,Team14,null,2025-12-31T23:23:07.000Z,null,2025-12-31T23:24:57.000Z,2025-12-31,110,86400,null,"Problem: Alarm Application Monitoring coupons Message: HTTPSConnectionPool(host='centraldoclienteIC04172locawebIC04172comIC04172br', port=443): Read timed out. (read timeout=25)",null,Monitoramento,false,Sem Intervenção,false,false,false,null,false
INC8654262,4,Baixa,NAO_CLASSIFICADO,NAO_CLASSIFICADO,null,Team14,IC00003,2025-12-31T23:23:05.000Z,null,2025-12-31T23:23:47.000Z,2025-12-31,42,86400,null,Problem: Check Application Monitoring,null,Monitoramento,false,Sem Intervenção,false,false,false,null,false
